# Chapter 2 — Triton Program Execution

Part 1: Foundations: what Triton is, how programs run, and how vectors are built.

**Thread.** One launch, one tile, one mask.

- The host chooses a grid, and each program receives one scalar coordinate per grid axis.
- `tl.program_id` turns that coordinate into a tile; `tl.arange` turns the tile into lanes.
- A mask removes dead lanes, and `tl.num_programs` lets one program walk several tiles.
- The same four moves compose into 1-D, 2-D, and 3-D work.

**Ladder.**

- **Concept:** grid coordinates and a CPU tile map.
- **Numeric demo:** the pid/offsets/mask table for `$N=10$`, `$B=4$`.
- **Micro-kernel:** 1-D vector add and 2-D transpose.
- **Applied use:** a grid-stride kernel where one program owns several tiles.

**Study tip.**

- Fix `$N=10$` and `BLOCK=4`.
- Write three columns by hand: `pid`, `offs = pid*4 + [0,1,2,3]`, and `mask = offs < 10`.
- The table has three rows. Program 2 has offsets `$8,9,10,11$`, mask `$T,T,F,F$`, and owns only `$8,9$`.
- If your table matches the worked example below, every kernel in this chapter is the same eight lines with a different pointer expression.

**Learning objectives**

- Read `tl.program_id(axis)` as a scalar grid coordinate in `$[0,P_\text{axis})$`, not as a thread id.
- Derive the launch grid for 1-D, 2-D, and 3-D work with `$P=\lceil N/B\rceil$` and the product rule.
- Map a program id to a tile and to matrix coordinates, including the inverse row-major map.
- Distinguish a Triton program (one CTA) from a CUDA thread and use `num_warps` correctly.
- Write a masked 1-D kernel and a masked 2-D kernel that cover every element exactly once.
- Write a grid-stride loop with `tl.num_programs(0)` and prove its coverage.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch01_introduction/01_complete.ipynb) · [Next →](../ch03_vector_construction/03_complete.ipynb)

**Estimated time:** 65–85 min · **Difficulty:** Beginner–Intermediate

**Prerequisites.**

- Chapter 1: host vs device code, `@triton.jit`, and launch syntax.
- Integer division and ceiling; basic matrix indexing.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

| § | Topic | Key formula | Shape/grid rule | Kernel/demo |
|---|---|---|---|---|
| **2.1** | `tl.program_id(0)` | `$i=pB+j$` | grid `(P,)`, `$p\in[0,P)$` | pid range demo |
| **2.2** | `tl.program_id(1)` | `$(r,c)=(p_mB_M+i,\ p_nB_N+j)$` | grid `(P_m,P_n)` | 2-D coordinate table |
| **2.3** | `tl.program_id(2)` | `$\text{addr}=bMN+rN+c$` | grid `(P_m,P_n,P_b)` | 3-D coordinate table |
| **2.4** | `tl.num_programs(0)` | `$\text{step}=P_0B$` | grid extent, not work size | grid-stride loop |
| **2.5** | 1-D program grid | `$P=\lceil N/B\rceil$` | one tile per program | vector add |
| **2.6** | 2-D program grid | `$P=(P_m,P_n)$` | two masks: rows and columns | transpose |
| **2.7** | 3-D program grid | `$P=(P_m,P_n,P_b)$` | batch on the slow axis | batched tile table |
| **2.8** | Program IDs and matrix coordinates | `$p_m=\lfloor t/G_n\rfloor,\ p_n=t\bmod G_n$` | linear label ↔ tile | inverse map demo |
| **2.9** | Programs vs CUDA threads and blocks | `$\text{threads/CTA}=32W$` | `num_warps` ≠ `BLOCK` | thread-count table |
| **2.10** | Launch-grid calculation | `$P=\lceil N/B\rceil=(N+B-1)//B$` | multiply per-axis counts | cdiv table |
| **2.11** | Multiple programs on one tensor | `$[0,N)=\bigsqcup_p I_p$` | reads may overlap, writes may not | ownership sets |
| **2.12** | Program-to-element mapping | four mappings in §12 | last axis is the lane axis | mapping comparison |

**Thesis.**

- The host chooses a grid; each program receives one scalar coordinate per axis.
- The kernel turns those coordinates into a tile and guards the tile with a mask.
- Everything else is pointer arithmetic.

In [ ]:
import math
import os
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, cdiv, require_gpu, run_if_gpu

# Back the opening study tip: N=10, BLOCK=4 -> P=3; program 2 owns 8 and 9.
STUDY_N, STUDY_BLOCK = 10, 4
study_offs = 2 * STUDY_BLOCK + torch.arange(STUDY_BLOCK)
study_mask = study_offs < STUDY_N
assert cdiv(STUDY_N, STUDY_BLOCK) == 3
assert study_offs.tolist() == [8, 9, 10, 11]
assert study_offs.dtype == torch.int64
assert study_mask.dtype == torch.bool
assert study_mask.tolist() == [True, True, False, False]
assert study_offs[study_mask].tolist() == [8, 9]
assert 32 * 4 == 128  # one CTA of four warps has 128 threads

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

## Concept diagrams

### 1-D, 2-D, and 3-D grids

```text
1-D: grid (P,)                 2-D: grid (Pm, Pn)               3-D: grid (Pm, Pn, Pb)
pid 0   pid 1   pid 2          pn=0   pn=1   pn=2             b=1  +-------------------+
  |       |       |            +------+------+------+               | (1,0,0)   (1,0,1) |
  v       v       v       pm=0 |(0,0) |(0,1) |(0,2) |               | (1,1,0)   (1,1,1) |
[0..3] [4..7] [8..9]           +------+------+------+               +-------------------+
                               |(1,0) |(1,1) |(1,2) |          b=0  +-------------------+
                               +------+------+------+               | (0,0,0)   (0,0,1) |
                               |(2,0) |(2,1) |(2,2) |               | (0,1,0)   (0,1,1) |
                               +------+------+------+               +-------------------+
```

- **Rule.** The 1-D picture is `$N=10$`, `BLOCK=4`, so `$P=3$`.
- The 2-D picture is the `$3\times3$` tile grid of the worked example.
- The 3-D picture adds a batch axis; the row and column axes keep their meaning.

### pid → tile → elements

```text
N = 10, BLOCK = 4, P = ceil(10/4) = 3

pid 0: offs [0 1 2 3]    mask [T T T T]  owns 0 1 2 3
pid 1: offs [4 5 6 7]    mask [T T T T]  owns 4 5 6 7
pid 2: offs [8 9 10 11]  mask [T T F F]  owns 8 9
```

- **Rule.** Program 0 owns `$0,1,2,3$`; program 1 owns `$4,5,6,7$`; program 2 owns `$8,9$`.
- Lanes 10 and 11 are dead lanes guarded by the mask.

### Programs vs CUDA blocks and threads

```text
Triton launch: grid (P,) of programs
   program 0 = one CTA    program 1 = one CTA    program 2 = one CTA
        |                      |                      |
        +-- num_warps=4 --> 4 warps × 32 threads = 128 CUDA threads per CTA
        +-- BLOCK=4 ------> only 4 logical lanes carry data

tl.program_id(0) is blockIdx.x, not threadIdx.x.
The compiler maps the logical lanes onto the CTA's threads.
```

- **Rule.** `tl.program_id(0)` is `blockIdx.x`, not `threadIdx.x`.
- With `num_warps=4`, a CTA has `$4\times32=128$` CUDA threads; `BLOCK=4` means only four logical lanes carry data.
- The compiler maps logical lanes onto the CTA's threads.

In [ ]:
# The diagrams claim P=3 for N=10/BLOCK=4, a 3x3 tile grid, a
# (3,3,2) batched grid, and 32*num_warps threads per CTA.
N, BLOCK = 10, 4
assert cdiv(N, BLOCK) == 3

M, BM = 5, 2
N2, BN = 7, 3
assert (cdiv(M, BM), cdiv(N2, BN)) == (3, 3)
assert (cdiv(M, BM), cdiv(N2, BN), 2) == (3, 3, 2)
assert 32 * 4 == 128

## Part I — Mathematics and concepts

### 1. `tl.program_id(0)`

- **Connection.** Builds on Chapter 1's host/device split and its launch syntax `kernel[grid](...)`.
- **Rule.** Axis 0 is the primary grid coordinate; every program sees a different scalar `$p$`, and every lane in that program sees the same `$p$` with a different `$j$`.
- **Formula.** `$p=\texttt{tl.program\_id}(0)\in\{0,\ldots,P-1\}$` and `$i(p,j)=pB+j$` for `$j\in\{0,\ldots,B-1\}$`.
- **Example.** `$N=10$`, `$B=4$` gives `$P=3$`; `$p=2$` starts at element `$8$`, so its first four offsets are `$8,9,10,11$`.
- **Where.** Flat tensors: elementwise maps, independent row reductions, and the outer loop of a streaming kernel.
- **Triton.** Use `offs = p * BLOCK + tl.arange(0, BLOCK)`; there is no `threadIdx` argument, and the grid tuple is host-side.
- **Failure.** Treating `$p$` as a global thread id, or computing the grid with `N // B`, which drops the tail.
- **Bridge.** The next section adds a second grid axis for matrix tiles.

In [ ]:
N, BLOCK = 10, 4
P = cdiv(N, BLOCK)

# §1 claims P=3, ids 0..2, and that program 2 starts at 8.
assert P == 3
assert list(range(P)) == [0, 1, 2]
for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    assert int(offs[0]) == pid * BLOCK
    assert int(offs[-1]) == pid * BLOCK + BLOCK - 1
assert (2 * BLOCK + torch.arange(BLOCK)).tolist() == [8, 9, 10, 11]

### 2. `tl.program_id(1)`

- **Connection.** Extends §1 from one scalar coordinate to a pair of coordinates.
- **Rule.** Axis 0 is the row tile and axis 1 is the column tile; the launch tuple and the kernel reads must agree.
- **Formula.** `$p_m=\texttt{tl.program\_id}(0)$`, `$p_n=\texttt{tl.program\_id}(1)$`, `$r=p_mB_M+i$`, `$c=p_nB_N+j$`, and `$\text{addr}(r,c)=rN+c$`.
- **Example.** `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$` gives grid `$(3,3)$` and nine programs; tile `$(0,2)$` keeps only column `$6$`, and tile `$(2,2)$` owns one cell.
- **Where.** Matrix copy, transpose, GEMM, and attention layouts.
- **Triton.** Launch `kernel[(cdiv(M,BM), cdiv(N,BN))](...)`; build `rows[:, None] * N + cols[None, :]` and mask with `(rows[:, None] < M) & (cols[None, :] < N)`. `tl.arange` needs a power-of-two length, so a pencil tile such as `$B_N=3$` is padded to 4 lanes and masked back.
- **Failure.** Swapping the axes transposes the tile assignment and usually reads out of bounds; masking only rows lets the last column tile write into the next row or past the buffer.
- **Bridge.** The next section adds the batch axis.

In [ ]:
M, N, BM, BN = 5, 7, 2, 3
Pm, Pn = cdiv(M, BM), cdiv(N, BN)

# §2 claims grid (3,3), nine programs, and partial corner tiles.
assert (Pm, Pn) == (3, 3)
assert Pm * Pn == 9

rows = 0 * BM + torch.arange(BM)
cols = 2 * BN + torch.arange(BN)
assert rows.tolist() == [0, 1]
assert cols[cols < N].tolist() == [6]

rows = 2 * BM + torch.arange(BM)
assert rows[rows < M].tolist() == [4]
assert cols[cols < N].tolist() == [6]

### 3. `tl.program_id(2)`

- **Connection.** Adds a slow batch axis to the row/column grid of §2.
- **Rule.** Axis 2 carries the slowest-varying batch: batch elements, attention heads, or images; any permutation is legal but must be documented and used consistently.
- **Formula.** `$p_b=\texttt{tl.program\_id}(2)$`, grid `$(P_m,P_n,P_b)$`, and `$\text{addr}(b,r,c)=bMN+rN+c$`.
- **Example.** `$B=2$`, `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$` gives grid `$(3,3,2)$`, 18 programs, and 70 live cells.
- **Where.** Batched GEMM, multi-head attention, and image tiles.
- **Triton.** Launch `(cdiv(M,BM), cdiv(N,BN), B)`, read `b = tl.program_id(2)`, and add `b * M * N` after the row/column address.
- **Failure.** An axis-order mismatch between host and kernel; a batch extent smaller than the batch count silently drops slices.
- **Bridge.** The next section reads the grid extent from inside the kernel.

In [ ]:
B, M, N, BM, BN = 2, 5, 7, 2, 3
Pm, Pn, Pb = cdiv(M, BM), cdiv(N, BN), B

# §3 claims grid (3,3,2), eighteen programs, and seventy live cells.
assert (Pm, Pn, Pb) == (3, 3, 2)
assert Pm * Pn * Pb == 18

total = 0
labels = []
for b in range(Pb):
    per_batch = []
    for pm in range(Pm):
        row = []
        for pn in range(Pn):
            rows = pm * BM + torch.arange(BM)
            cols = pn * BN + torch.arange(BN)
            total += len(rows[rows < M]) * len(cols[cols < N])
            row.append((b * Pm + pm) * Pn + pn)
        per_batch.append(row)
    labels.append(per_batch)

assert total == B * M * N == 70
assert labels[0] == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
assert labels[1] == [[9, 10, 11], [12, 13, 14], [15, 16, 17]]

### 4. `tl.num_programs(0)`

- **Connection.** §1–§3 read coordinates; this section reads the grid extent itself.
- **Rule.** `tl.num_programs(axis)` is the launch extent of one axis inside the kernel; it is a scalar constant for the launch, unlike `tl.program_id(axis)`, which varies per program.
- **Formula.** `$P_0=\texttt{tl.num\_programs}(0)$`, grid-stride step `$P_0B$`, and software tile label `$t=p_mP_n+p_n$` for a 2-D grid.
- **Example.** Grids `$(3,)$`, `$(3,3)$`, `$(3,3,2)$` report products `$3,9,18$`; with `$N=10$`, `$B=4$`, `$G=2$`, the stride is `$8$`, so program 0 starts at `$0,8$` and program 1 at `$4$`.
- **Where.** Grid-stride loops, persistent kernels, and linear tile labels.
- **Triton.** Compare `tl.num_programs(axis)` with `tl.program_id(axis)`; do not pass `$P$` as a runtime argument when the launch already knows it.
- **Failure.** Confusing the grid size with the problem size (`num_programs(0)` is not `$N$` and not `$B$`); a grid-stride step of `$B$` instead of `$P_0B$` re-processes tiles.
- **Bridge.** The next section fixes the 1-D decomposition.

In [ ]:
grids = [(3,), (3, 3), (3, 3, 2)]
assert [math.prod(g) for g in grids] == [3, 9, 18]
assert [len(g) for g in grids] == [1, 2, 3]

N, BLOCK, programs = 10, 4, 2
stride = programs * BLOCK
assert stride == 8
assert list(range(0, N, stride)) == [0, 8]
assert list(range(BLOCK, N, stride)) == [4]
assert N // BLOCK == 2  # the wrong grid size, shown for contrast

### 5. 1-D program grid

- **Connection.** Uses the grid extent from §4 to partition a flat vector.
- **Rule.** Program `$p$` owns the contiguous tile `$[pB,(p+1)B)$`, clipped at `$N$`.
- **Formula.** `$P=\lceil N/B\rceil$` and `$I_p=[pB,(p+1)B)\cap[0,N)$`.
- **Example.** `$N=10$`, `$B=4$` gives `$P=3$`; `$N//B=2$` would launch only two programs and leave `$8,9$` unwritten.
- **Where.** The default decomposition for a flat vector; lane `$j$` touches `$pB+j$`, so adjacent lanes touch adjacent words.
- **Triton.** Launch `grid=(triton.cdiv(N, BLOCK),)`; use `offs = p*BLOCK + tl.arange(0, BLOCK)` and `mask = offs < n_elements` on every load and store.
- **Failure.** `$P=N//B$` drops the partial tail; omitting the mask reads or writes up to `$B-1$` elements past the buffer.
- **Bridge.** The next section adds a second dimension.

In [ ]:
N, BLOCK = 10, 4
P = cdiv(N, BLOCK)

assert N // BLOCK == 2   # truncation drops the tail
assert P == 3            # the correct ceiling

expected_offs = [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
expected_mask = [[True] * 4, [True] * 4, [True, True, False, False]]
expected_owned = [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]

for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    mask = offs < N
    assert offs.tolist() == expected_offs[pid]
    assert mask.tolist() == expected_mask[pid]
    assert offs[mask].tolist() == expected_owned[pid]

print("§5: pid/offs/mask table asserted; N//BLOCK would miss 8 and 9.")

### 6. 2-D program grid

- **Connection.** Lifts the 1-D contiguous tile of §5 to a rectangle.
- **Rule.** Two independent tile counts; a rectangular tile needs both a row mask and a column mask.
- **Formula.** `$P_m=\lceil M/B_M\rceil$`, `$P_n=\lceil N/B_N\rceil$`, grid `$(P_m,P_n)$`, total `$P_mP_n$`; program `$(p_m,p_n)$` owns rows `$p_mB_M,\ldots,p_mB_M+B_M-1$` and columns `$p_nB_N,\ldots,p_nB_N+B_N-1$`, clipped at `$M$` and `$N$`.
- **Example.** `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$` gives `$(3,3)$` and nine programs; the live-cell count grid is `$[[6,6,2],[6,6,2],[3,3,1]]$`, summing to `$35$`.
- **Where.** Matrix copy, transpose, GEMM, and any rectangular tile.
- **Triton.** Build `rows = p_m*BM + tl.arange(0,BM)`, `cols = p_n*BN + tl.arange(0,BN)`, and `mask = (rows[:,None] < M) & (cols[None,:] < N)`; both extents use `triton.cdiv`.
- **Failure.** Forgetting one of the two masks; launching `(Pm,)` but reading `program_id(1)` (always 0); using matrix width `$N$` where tile count `$P_n$` belongs.
- **Bridge.** The next section adds the batch axis.

In [ ]:
M, N, BM, BN = 5, 7, 2, 3
Pm, Pn = cdiv(M, BM), cdiv(N, BN)

assert (Pm, Pn) == (3, 3)
counts = []
for pm in range(Pm):
    for pn in range(Pn):
        rows = pm * BM + torch.arange(BM)
        cols = pn * BN + torch.arange(BN)
        counts.append(len(rows[rows < M]) * len(cols[cols < N]))

count_grid = torch.tensor(counts).view(Pm, Pn)
assert count_grid.tolist() == [[6, 6, 2], [6, 6, 2], [3, 3, 1]]
assert count_grid.sum().item() == M * N == 35

### 7. 3-D program grid

- **Connection.** Adds a batch axis to §6's 2-D tile grid.
- **Rule.** Batch and head axes are independent of the matrix tile axes; one launch can cover a batch without folding it into the row index.
- **Formula.** Grid `$(P_m,P_n,P_b)$` with `$P_b=B$` and `$P_mP_nP_b$` programs; with §3's convention, `$t=(bP_m+p_m)P_n+p_n$` and `$\text{addr}=bMN+rN+c$`.
- **Example.** `$B=2$`, `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$` gives `$(3,3,2)$` and 18 programs; batch 0 has labels `$0,\ldots,8$` and batch 1 has `$9,\ldots,17$`.
- **Where.** Batched GEMM, multi-head attention, and image tiles.
- **Triton.** Launch `(cdiv(M,BM), cdiv(N,BN), B)`, read `b = tl.program_id(2)`, and add `b*M*N` to the 2-D offset; keep `$B$` runtime if it can change.
- **Failure.** Axis-order mismatch between host and kernel; computing `b*M*N` but leaving the batch term out of the pointer, so the kernel rewrites batch 0.
- **Bridge.** The next section decodes a flat program id into tile coordinates.

In [ ]:
B, M, N, BM, BN = 2, 5, 7, 2, 3
Pm, Pn, Pb = cdiv(M, BM), cdiv(N, BN), B

assert (Pm, Pn, Pb) == (3, 3, 2)
assert Pm * Pn * Pb == 18

labels = [
    [[(b * Pm + pm) * Pn + pn for pn in range(Pn)] for pm in range(Pm)]
    for b in range(Pb)
]
assert labels == [
    [[0, 1, 2], [3, 4, 5], [6, 7, 8]],
    [[9, 10, 11], [12, 13, 14], [15, 16, 17]],
]
assert B * M * N == 70

### 8. Program IDs and matrix coordinates

- **Connection.** §6–§7 launch 2-D directly; this section recovers coordinates when the launch is flat.
- **Rule.** A row-major tile label `$t$` maps to `$(p_m,p_n)$` by division and remainder; `$t$` is a software label, not the hardware's block-scheduling order.
- **Formula.** `$p_m=\lfloor t/G_n\rfloor$`, `$p_n=t\bmod G_n$`, inverse `$t=p_mG_n+p_n$`; element coordinates `$r=p_mB_M+i$`, `$c=p_nB_N+j$`, flat address `$rN+c$`.
- **Example.** For `$G_m=G_n=3$`, `$t=7$` maps to `$(2,1)$` and back to `$7$`; the nine labels are `$0,\ldots,8$`.
- **Where.** 1-D launches for 2-D work, persistent kernels, and bucketed schedules.
- **Triton.** Use `program_id(0/1)` when the grid is already 2-D; otherwise pass `$G_n$` as a `tl.constexpr` or read `tl.num_programs(1)`.
- **Failure.** Dividing by matrix width `$N$` instead of column-tile count `$G_n$`; swapping quotient and remainder; flattening with `$G_m$` instead of `$G_n$`.
- **Bridge.** The next section separates the program from the CUDA thread.

In [ ]:
Gm, Gn = 3, 3
assert Gm * Gn == 9

for t in range(Gm * Gn):
    pm, pn = t // Gn, t % Gn
    assert 0 <= pm < Gm and 0 <= pn < Gn
    assert pm * Gn + pn == t

assert (7 // 3, 7 % 3) == (2, 1)
assert [pm * Gn + pn for pm in range(Gm) for pn in range(Gn)] == list(range(9))

### 9. Programs vs CUDA threads and blocks

- **Connection.** §1–§8 treat the program as the unit; Chapter 1's launch syntax now needs the CTA/thread vocabulary.
- **Rule.** One Triton program is one CUDA CTA (block); a tile is a logical vector, and the compiler distributes its lanes over the CTA's threads.
- **Formula.** Threads per CTA `$32W$`; average elements per thread `$\text{tile elements}/(32W)$`; for a 1-D tile `$B/(32W)$`, for a 2-D tile `$B_MB_N/(32W)$`.
- **Example.** `$B=256$`, `$W=4$` gives 128 threads and about 2 elements per thread; a `$2\times3$` tile with one warp gives `$6/32=0.1875$`; `$N=10$`, `$B=4$`, `$W=4$` launches three CTAs and 384 threads.
- **Where.** Cost model, occupancy, and the difference between `blockIdx` and `threadIdx`.
- **Triton.** `num_warps` is a launch meta-parameter, not a kernel argument; `BLOCK` controls logical tile size, and `num_warps` controls how many threads execute it.
- **Failure.** Equating `BLOCK` with thread count; sizing the grid by CUDA threads, where a grid of `$N$` programs is `$N$` CTAs, not `$N$` threads.
- **Bridge.** The next section turns these pieces into a launch-grid formula.

In [ ]:
for warps in (1, 2, 4, 8):
    threads = 32 * warps
    assert threads == 32 * warps
    assert abs(256 / threads - 8 / warps) < 1e-12

assert 32 * 4 == 128
assert 256 / 128 == 2.0
assert abs((2 * 3) / 32 - 0.1875) < 1e-12

N, BLOCK, warps = 10, 4, 4
P = cdiv(N, BLOCK)
assert P * 32 * warps == 384

### 10. Launch-grid calculation

- **Connection.** Combines §5's 1-D ceiling, §6's product rule, and §9's CTA model into the host-side grid.
- **Rule.** Launch enough programs to cover every element; over-covering is safe when the tail is masked, and under-covering is a silent correctness bug.
- **Formula.** For `$N\gt 0$`, `$P=\lceil N/B\rceil=\lfloor(N-1)/B\rfloor+1=(N+B-1)//B$`; for 2-D and 3-D work, `$P=P_mP_n$` and `$P=P_mP_nP_b$`.
- **Example.** `$(N,B)$` pairs `$(10,4),(12,4),(13,4),(1,4),(0,4)$` give `$3,3,4,1,0$`; `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$` gives `$3\cdot3=9$`.
- **Where.** The first line of every host wrapper that launches a Triton kernel.
- **Triton.** Use `triton.cdiv(N, BLOCK)` or `ttools.cdiv`; for a zero-size tensor launch `(0,)` or return early.
- **Failure.** Integer truncation `N//B` loses the tail; adding one unconditionally launches a fully masked program when `$B$` divides `$N$`; passing a float as a grid extent raises.
- **Bridge.** The next section asks who may touch which element.

In [ ]:
cases = {(10, 4): 3, (12, 4): 3, (13, 4): 4, (1, 4): 1, (0, 4): 0}
for (N, B), expected in cases.items():
    assert cdiv(N, B) == expected
    assert (N - 1) // B + 1 == expected

M, N, BM, BN = 5, 7, 2, 3
assert cdiv(M, BM) * cdiv(N, BN) == 9

print("§10: ceiling and product rules asserted on five 1-D cases and one 2-D case.")

### 11. Multiple programs on one tensor

- **Connection.** §10 sizes the grid; this section assigns ownership of the tensor.
- **Rule.** A launch partitions the index set; reads may overlap safely, but writes need one owner or atomics.
- **Formula.** `$[0,N)=\bigsqcup_{p=0}^{P-1} I_p$` with `$I_p=[pB,(p+1)B)\cap[0,N)$`.
- **Example.** `$N=10$`, `$B=4$` partitions into `$\{0,1,2,3\}$`, `$\{4,5,6,7\}$`, `$\{8,9\}$`; two programs may read `x[4:8]`, but two writes to `x[4:8]` leave the last writer's value.
- **Where.** Elementwise kernels need a partition, not a replication; shared tables are safe to read.
- **Triton.** There is no inter-program synchronization and no ordering guarantee between `pid` values; use `tl.atomic_add` for a shared accumulator, give each output exactly one owner, or launch a second kernel as a barrier.
- **Failure.** Two programs writing the same index ("last writer wins"); a read-after-write dependency without a barrier; launching the same tile twice when the grid is too large and the store is unmasked.
- **Bridge.** The next section chooses the mapping that enforces this ownership.

In [ ]:
N, BLOCK = 10, 4
P = cdiv(N, BLOCK)

owners = [set(range(pid * BLOCK, min((pid + 1) * BLOCK, N))) for pid in range(P)]
assert sorted(set().union(*owners)) == list(range(N))
for p in range(P):
    for q in range(p + 1, P):
        assert owners[p].isdisjoint(owners[q])

x = torch.arange(N)
assert torch.equal(x[4:8], x[4:8])

out = torch.full((N,), -1, dtype=torch.int64)
out[4:8] = 0
out[4:8] = 1
assert out[4:8].tolist() == [1, 1, 1, 1]  # last writer wins

### 12. Program-to-element mapping

- **Connection.** §11 requires unique ownership; this section chooses how program `$p$` and lane `$j$` map to an element.
- **Rule.** The mapping is a design choice, not a Triton property; it determines coalescing, load balance, and which masks are needed.
- **Formula.**

| Mapping | Index owned by program `$p$`, lane `$j$` | Use |
|---|---|---|
| block-contiguous | `$i=pB+j$` | 1-D elementwise |
| grid-stride | `$i=(p+kP)B+j$`, `$k=0,1,\ldots$` | persistent / small grid |
| row-per-program | `$i=pN+j$` (with `$B\ge N$`) | row-wise normalization |
| 2-D tile | `$i=(p_mB_M+i_r)N+(p_nB_N+j_c)$` | matrix tiles |

- **Example.** `$N=10$`, `$B=4$` gives block ownership `$[0,0,0,0,1,1,1,1,2,2]$`; a grid-stride launch with `$G=2$` gives `$[0,0,0,0,1,1,1,1,0,0]$`; row-per-program for `$M=5$`, `$N=7$` starts rows at `$0,7,14,21,28$`; a 2-D tile with `$p_m=p_n=1$`, `$i=j=0$` has `$r=2$`, `$c=3$`, address `$17$`.
- **Where.** Contiguous lanes for coalescing, and the last matrix axis as the lane axis.
- **Triton.** Keep `tl.arange` on the contiguous axis; for a 2-D tile use `rows[:, None] * N + cols[None, :]`, never `p_m*BM + p_n*BN + tl.arange(...)`.
- **Failure.** Flattening with tile widths (`rows*BN + cols`) instead of matrix width (`rows*N + cols`); a grid-stride step of `$B$` instead of `$PB$`.
- **Bridge.** The worked example now fixes every value.
- **Where this leads.** [Chapter 3 — Triton Vector Construction](../ch03_vector_construction/03_complete.ipynb) builds vectors with `tl.arange`, `tl.full`, `tl.zeros`, and `tl.cast`; the pid machinery here supplies the tile coordinates those constructors fill.

In [ ]:
N, BLOCK = 10, 4
P = cdiv(N, BLOCK)

owner_block = [pid for pid in range(P) for _ in range(min(BLOCK, N - pid * BLOCK))]
assert owner_block == [0, 0, 0, 0, 1, 1, 1, 1, 2, 2]

owner_stride = [-1] * N
for pid in range(2):
    for t in range(pid, P, 2):
        for i in range(t * BLOCK, min((t + 1) * BLOCK, N)):
            owner_stride[i] = pid
assert owner_stride == [0, 0, 0, 0, 1, 1, 1, 1, 0, 0]

M, N = 5, 7
assert [r * N for r in range(M)] == [0, 7, 14, 21, 28]

BM, BN = 2, 3
pm, pn, ir, jc = 1, 1, 0, 0
r, c = pm * BM + ir, pn * BN + jc
assert (r, c) == (2, 3)
assert r * N + c == 17

## Worked numerical example

### 1-D: `$N=10$`, `$B=4$`

- **Rule.** `$P=\lceil10/4\rceil=3$`.
- **Table.**

| `pid` | `offs` | `mask` | elements owned |
|---:|---|---|---|
| 0 | 0,1,2,3 | T T T T | 0,1,2,3 |
| 1 | 4,5,6,7 | T T T T | 4,5,6,7 |
| 2 | 8,9,10,11 | T T F F | 8,9 |

- **Vector.** The same ownership picture, with `·` for a dead lane:

$$
x=(0,1,2,3,\;4,5,6,7,\;8,9,\cdot,\cdot),
\qquad
\underbrace{0\,1\,2\,3}_{p=0}\;
\underbrace{4\,5\,6\,7}_{p=1}\;
\underbrace{8\,9}_{p=2}.
$$

- **Ownership.** The last program has two dead lanes. Its arithmetic may execute on lanes 10 and 11, but its load and store are masked, so the output has exactly 10 elements.

### 2-D: `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$`

- **Rule.** `$P_m=\lceil5/2\rceil=3$`, `$P_n=\lceil7/3\rceil=3$`, grid `$(3,3)$`, nine programs.
- **Table.**

| `$p_m$` | `$p_n$` | rows | cols | valid rows | valid cols | elems |
|---:|---:|---|---|---|---|---:|
| 0 | 0 | 0,1 | 0,1,2 | 0,1 | 0,1,2 | 6 |
| 0 | 1 | 0,1 | 3,4,5 | 0,1 | 3,4,5 | 6 |
| 0 | 2 | 0,1 | 6,7,8 | 0,1 | 6 | 2 |
| 1 | 0 | 2,3 | 0,1,2 | 2,3 | 0,1,2 | 6 |
| 1 | 1 | 2,3 | 3,4,5 | 2,3 | 3,4,5 | 6 |
| 1 | 2 | 2,3 | 6,7,8 | 2,3 | 6 | 2 |
| 2 | 0 | 4,5 | 0,1,2 | 4 | 0,1,2 | 3 |
| 2 | 1 | 4,5 | 3,4,5 | 4 | 3,4,5 | 3 |
| 2 | 2 | 4,5 | 6,7,8 | 4 | 6 | 1 |

- **Counts.** The counts sum to `$35=5\cdot7$`; the tile label is `$t=p_m\cdot3+p_n$`, so the nine labels are `$0,\ldots,8$`.
- **Matrix.** As a `$5\times7$` matrix, entry `$T_{rc}$` is the tile that owns cell `$(r,c)$`:

$$
T=
\begin{bmatrix}
0&0&0&1&1&1&2\\
0&0&0&1&1&1&2\\
3&3&3&4&4&4&5\\
3&3&3&4&4&4&5\\
6&6&6&7&7&7&8
\end{bmatrix},
\qquad
C=
\begin{bmatrix}
6&6&2\\
6&6&2\\
3&3&1
\end{bmatrix}.
$$

- **Assertion.** `$C_{p_mp_n}$` is the number of live cells in tile `$(p_m,p_n)$`; the notebook asserts `$T$`, `$C$`, and `$\sum C=35$`, and does not print them.

In [ ]:
def tile_table_1d(N: int, BLOCK: int):
    """Return P and one row per program: offs, mask, owned elements."""
    P = cdiv(N, BLOCK)
    rows = []
    for pid in range(P):
        offs = pid * BLOCK + torch.arange(BLOCK)
        mask = offs < N
        rows.append({
            "pid": pid,
            "offs": offs.tolist(),
            "mask": mask.tolist(),
            "owned": offs[mask].tolist(),
        })
    return P, rows


def tile_table_2d(M: int, N: int, BM: int, BN: int):
    """Return (Pm, Pn) and one row per tile: coordinates, valid ranges, count."""
    Pm, Pn = cdiv(M, BM), cdiv(N, BN)
    rows = []
    for pm in range(Pm):
        for pn in range(Pn):
            r = pm * BM + torch.arange(BM)
            c = pn * BN + torch.arange(BN)
            valid_r = r[r < M].tolist()
            valid_c = c[c < N].tolist()
            rows.append({
                "pm": pm,
                "pn": pn,
                "t": pm * Pn + pn,
                "rows": r.tolist(),
                "cols": c.tolist(),
                "valid_rows": valid_r,
                "valid_cols": valid_c,
                "count": len(valid_r) * len(valid_c),
            })
    return (Pm, Pn), rows


def _check_worked_example() -> None:
    """Assert the 1-D table, the 2-D tile map, and the 5x7 label matrix."""
    P1, rows1 = tile_table_1d(10, 4)
    assert P1 == 3
    assert [r["offs"] for r in rows1] == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
    assert [r["mask"] for r in rows1] == [
        [True, True, True, True],
        [True, True, True, True],
        [True, True, False, False],
    ]
    assert [r["owned"] for r in rows1] == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]

    (Pm, Pn), rows2 = tile_table_2d(5, 7, 2, 3)
    assert (Pm, Pn) == (3, 3)
    assert [r["t"] for r in rows2] == list(range(9))
    assert [r["count"] for r in rows2] == [6, 6, 2, 6, 6, 2, 3, 3, 1]
    assert [r["valid_rows"] for r in rows2] == [
        [0, 1], [0, 1], [0, 1], [2, 3], [2, 3], [2, 3], [4], [4], [4],
    ]
    assert [r["valid_cols"] for r in rows2] == [
        [0, 1, 2], [3, 4, 5], [6], [0, 1, 2], [3, 4, 5], [6],
        [0, 1, 2], [3, 4, 5], [6],
    ]

    counts = torch.tensor([r["count"] for r in rows2], dtype=torch.int64).view(Pm, Pn)
    assert counts.tolist() == [[6, 6, 2], [6, 6, 2], [3, 3, 1]]
    assert counts.sum().item() == 35

    label_matrix = torch.zeros((5, 7), dtype=torch.int64)
    for row in rows2:
        for r in row["valid_rows"]:
            for c in row["valid_cols"]:
                label_matrix[r, c] = row["t"]
    assert label_matrix.tolist() == [
        [0, 0, 0, 1, 1, 1, 2],
        [0, 0, 0, 1, 1, 1, 2],
        [3, 3, 3, 4, 4, 4, 5],
        [3, 3, 3, 4, 4, 4, 5],
        [6, 6, 6, 7, 7, 7, 8],
    ]


_check_worked_example()
print("Worked example: 1-D table, 5x7 tile-label matrix, and count grid verified.")

In [ ]:
def test_worked_example() -> None:
    """The final dispatch re-runs the worked-example assertions."""
    _check_worked_example()

## Part II — Method 01: naive CPU simulation of the grid

- **Connection.** The worked example fixes the map; Method 01 executes that map with Python loops.
- **Rule.** One iteration per program, one per lane; the `if` is the mask.
- **Code.**

```python
for pid in range(P):          # one Triton program
    for j in range(BLOCK):    # one logical lane
        i = pid * BLOCK + j
        if i < N:
            out[i] = value(pid, j)
```

- **Why.** It is intentionally slow; its job is to make the mapping explicit and to provide the reference.
- **Bridge.** Method 02 replaces the loops with a launch and the `if` with `mask = offs < N`.

In [ ]:
def naive_grid_1d(N: int, BLOCK: int):
    """Method 01: one Python iteration per program and per lane."""
    out = torch.full((N,), -1, dtype=torch.int64)
    owner = torch.full((N,), -1, dtype=torch.int64)
    for pid in range(cdiv(N, BLOCK)):
        for j in range(BLOCK):
            i = pid * BLOCK + j
            if i < N:
                out[i] = pid
                owner[i] = pid
    return out, owner


def naive_grid_2d(M: int, N: int, BM: int, BN: int):
    """Method 01 for a matrix: each cell stores its tile label pm*Pn+pn."""
    Pm, Pn = cdiv(M, BM), cdiv(N, BN)
    out = torch.full((M, N), -1, dtype=torch.int64)
    owner = torch.full((M, N), -1, dtype=torch.int64)
    for pm in range(Pm):
        for pn in range(Pn):
            label = pm * Pn + pn
            for i in range(BM):
                for j in range(BN):
                    r, c = pm * BM + i, pn * BN + j
                    if r < M and c < N:
                        out[r, c] = label
                        owner[r, c] = label
    return out, owner


out1, owner1 = naive_grid_1d(10, 4)
assert out1.tolist() == [0, 0, 0, 0, 1, 1, 1, 1, 2, 2]
assert owner1.tolist() == out1.tolist()

tiles, owner2 = naive_grid_2d(5, 7, 2, 3)
assert tiles.tolist() == [
    [0, 0, 0, 1, 1, 1, 2],
    [0, 0, 0, 1, 1, 1, 2],
    [3, 3, 3, 4, 4, 4, 5],
    [3, 3, 3, 4, 4, 4, 5],
    [6, 6, 6, 7, 7, 7, 8],
]
assert bool((tiles == owner2).all().item())
assert bool((tiles >= 0).all().item())
assert tiles.numel() == 35

print("Method 01: the Python grid simulation reproduces both markdown tables.")

In [ ]:
def test_naive_grid() -> None:
    """The CPU simulation matches the hand-built tables."""
    out, owner = naive_grid_1d(10, 4)
    assert out.tolist() == [0, 0, 0, 0, 1, 1, 1, 1, 2, 2]
    assert owner.tolist() == out.tolist()

    tiles, owner2 = naive_grid_2d(5, 7, 2, 3)
    assert tiles.tolist() == [
        [0, 0, 0, 1, 1, 1, 2],
        [0, 0, 0, 1, 1, 1, 2],
        [3, 3, 3, 4, 4, 4, 5],
        [3, 3, 3, 4, 4, 4, 5],
        [6, 6, 6, 7, 7, 7, 8],
    ]
    assert bool((tiles == owner2).all().item())
    assert int((tiles < 0).sum().item()) == 0
    assert int(torch.bincount(tiles.flatten(), minlength=9).sum().item()) == 35

## Part III — Method 02: a 1-D Triton kernel

- **Connection.** Replaces Method 01's Python loops with one masked Triton tile per program.
- **Rule.** Both loads use the same mask; tail lanes load `other=0.0`, add zero, and are discarded by the masked store.
- **Formula.** `$\text{offs}=pB+\{0,\ldots,B-1\}$`, `$\text{mask}=\text{offs}\lt N$`, `$\text{out}[\text{offs}]=x[\text{offs}]+y[\text{offs}]$`.
- **Example.** The tile map is the `$N=10$`, `$B=4$` table from the worked example; the next cell asserts it again before defining the kernel.
- **Why.** A vector add is the smallest kernel with two reads and one write; the grid and the mask are the only things that can be wrong.
- **Bridge.** Method 03 adds a second coordinate and a second mask.

In [ ]:
N, BLOCK = 10, 4
P = cdiv(N, BLOCK)
assert P == 3
expected_offs = [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
expected_mask = [[True] * 4, [True] * 4, [True, True, False, False]]
for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    assert offs.tolist() == expected_offs[pid]
    assert (offs < N).tolist() == expected_mask[pid]


@triton.jit
def m02_vec_add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + y, mask=mask)


def vec_add(x, y, BLOCK: int = 4):
    """Method 02: launch ceil(N/BLOCK) programs, one masked tile each."""
    x = torch.as_tensor(x).contiguous()
    y = torch.as_tensor(y).contiguous()
    assert x.shape == y.shape, (x.shape, y.shape)
    out = torch.empty_like(x)
    m02_vec_add_kernel[(cdiv(x.numel(), BLOCK),)](
        x, y, out, x.numel(), BLOCK=BLOCK
    )
    return out

print("Method 02: the 1-D tile map is asserted before the launch.")

In [ ]:
def test_vec_add_kernel() -> None:
    require_gpu("m02 vec add")
    x = torch.arange(10, device=DEVICE, dtype=torch.float32)
    y = torch.full_like(x, 100.0)
    check("vec add N=10 BLOCK=4", vec_add(x, y, BLOCK=4), x + y)

    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    y = torch.randn(1000, device=DEVICE)
    check("vec add N=1000 BLOCK=64", vec_add(x, y, BLOCK=64), x + y)

## Part IV — Method 03: a 2-D kernel that maps ids to matrix tiles

- **Connection.** Lifts Method 02's flat `offs` to row and column coordinates.
- **Rule.** The mask is the outer product of the row and column masks.
- **Formula.** `$r=p_mB_M+i$`, `$c=p_nB_N+j$`, `$\text{addr}=rN+c$`, and `$\text{mask}_{ij}=[r_i\lt M]\land[c_j\lt N]$`.
- **Example.** A transpose reads `A[r,c]` and writes `B[c,r]`, so the store address is `c*M+r`; the grid is `$(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$`.
- **Padding.** `tl.arange(0,k)` requires a power-of-two `$k$`; the pencil `$B_N=3$` is padded to `$B_N=4$`, so the grid becomes `$(3,2)$` instead of `$(3,3)$` and column 7 is masked by `cols < N`. Padding moves the tile boundaries and the grid, but coverage is unchanged: all 35 cells are still owned exactly once.
- **Table.** For `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=4$`, there are six programs:

| `$p_m$` | `$p_n$` | rows | cols | valid rows | valid cols | elems |
|---:|---:|---|---|---|---|---:|
| 0 | 0 | 0,1 | 0,1,2,3 | 0,1 | 0,1,2,3 | 8 |
| 0 | 1 | 0,1 | 4,5,6,7 | 0,1 | 4,5,6 | 6 |
| 1 | 0 | 2,3 | 0,1,2,3 | 2,3 | 0,1,2,3 | 8 |
| 1 | 1 | 2,3 | 4,5,6,7 | 2,3 | 4,5,6 | 6 |
| 2 | 0 | 4,5 | 0,1,2,3 | 4 | 0,1,2,3 | 4 |
| 2 | 1 | 4,5 | 4,5,6,7 | 4 | 4,5,6 | 3 |

- **Bridge.** The next cell asserts this padded map, then defines the kernel.

In [ ]:
M, N, BM, BN = 5, 7, 2, 4
Pm, Pn = cdiv(M, BM), cdiv(N, BN)
assert (Pm, Pn) == (3, 2)

counts = []
valid_cols_01 = None
for pm in range(Pm):
    for pn in range(Pn):
        rows = pm * BM + torch.arange(BM)
        cols = pn * BN + torch.arange(BN)
        counts.append(len(rows[rows < M]) * len(cols[cols < N]))
        if (pm, pn) == (0, 1):
            valid_cols_01 = cols[cols < N].tolist()

assert counts == [8, 6, 8, 6, 4, 3]
assert sum(counts) == 35
assert valid_cols_01 == [4, 5, 6]


@triton.jit
def m02_transpose_kernel(a_ptr, b_ptr, M, N, BM: tl.constexpr, BN: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)

    mask = (rows[:, None] < M) & (cols[None, :] < N)
    tile = tl.load(a_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    tl.store(b_ptr + cols[None, :] * M + rows[:, None], tile, mask=mask)


def transpose2d(a, BM: int = 2, BN: int = 4):
    """Method 03: B[c, r] = A[r, c] with a 2-D grid and two boundary masks."""
    a = torch.as_tensor(a).contiguous()
    M, N = a.shape
    out = torch.empty((N, M), dtype=a.dtype, device=a.device)
    m02_transpose_kernel[(cdiv(M, BM), cdiv(N, BN))](a, out, M, N, BM=BM, BN=BN)
    return out

print("Method 03: the padded 2x4 map is asserted; the launch grid is (3,2).")

In [ ]:
def test_transpose_kernel() -> None:
    require_gpu("m02 transpose")
    x = torch.arange(35, device=DEVICE, dtype=torch.float32).view(5, 7)
    check("transpose 5x7 with 2x4 tiles", transpose2d(x, BM=2, BN=4), x.t())

    torch.manual_seed(0)
    x = torch.randn(33, 100, device=DEVICE)
    check("transpose 33x100 with 8x16 tiles", transpose2d(x, BM=8, BN=16), x.t())

## `tl.num_programs` and grid-stride processing

- **Connection.** §4 read the launch extent; this section uses it so one program can own several tiles.
- **Rule.** A grid of `$G$` programs need not equal the `$P$` tiles; program `$p$` walks `$t=p,p+G,p+2G,\ldots\lt P$`, and the element step is `$GB$`, not `$B$`.
- **Coverage.** Every tile is visited exactly once because the residues `$t\bmod G$` are distinct.
- **Example.** For `$N=10$`, `$B=4$`, `$P=3$`, and `$G=2$`:

| program | tiles | starts | elements |
|---:|---|---|---|
| 0 | 0, 2 | 0, 8 | 0–3, 8–9 |
| 1 | 1 | 4 | 4–7 |

- **Labels.** The next cell asserts this table and defines a kernel that writes a per-program, per-iteration label; program 0 writes 0 then 1, and program 1 writes 10.
- **Bridge.** The final tests run the CPU and GPU paths.

In [ ]:
N, BLOCK, PROGRAMS = 10, 4, 2
n_tiles = cdiv(N, BLOCK)
assert n_tiles == 3
assert list(range(0, n_tiles, PROGRAMS)) == [0, 2]
assert [t * BLOCK for t in range(0, n_tiles, PROGRAMS)] == [0, 8]
assert list(range(1, n_tiles, PROGRAMS)) == [1]
assert [t * BLOCK for t in range(1, n_tiles, PROGRAMS)] == [4]

expected_labels = torch.tensor([0, 0, 0, 0, 10, 10, 10, 10, 1, 1], dtype=torch.int32)
assert expected_labels[:4].unique().tolist() == [0]
assert expected_labels[4:8].unique().tolist() == [10]
assert expected_labels[8:].unique().tolist() == [1]


@triton.jit
def m02_grid_stride_kernel(out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    nprog = tl.num_programs(0)
    n_tiles = (n_elements + BLOCK - 1) // BLOCK

    t = pid
    k = 0
    while t < n_tiles:
        start = t * BLOCK
        offs = start + tl.arange(0, BLOCK)
        tl.store(out_ptr + offs, pid * 10 + k, mask=offs < n_elements)
        t += nprog
        k += 1


def grid_stride_labels(n: int = 10, BLOCK: int = 4, programs: int = 2):
    out = torch.full((n,), -1, dtype=torch.int32, device=DEVICE)
    m02_grid_stride_kernel[(programs,)](out, n, BLOCK=BLOCK)
    return out

print("Grid-stride: 3 tiles covered by 2 programs with step 8.")

In [ ]:
def test_grid_stride_kernel() -> None:
    require_gpu("m02 grid stride")
    out = grid_stride_labels()
    expected = torch.tensor(
        [0, 0, 0, 0, 10, 10, 10, 10, 1, 1], device=DEVICE, dtype=torch.int32
    )
    check("grid-stride labels N=10 BLOCK=4 programs=2", out, expected)
    assert bool((out >= 0).all().item()), "every element must be written"

    expected_owner = torch.tensor(
        [0, 0, 0, 0, 1, 1, 1, 1, 0, 0], device=DEVICE, dtype=torch.int32
    )
    owner = torch.where(out >= 10, 1, 0).to(torch.int32)
    check("grid-stride owner vector", owner, expected_owner)

## Pitfalls

- **Rule.** Each row is a concrete failure mode and its smallest fix.

| Pitfall | What happens | Fix |
|---|---|---|
| off-by-one tail | `offs <= N` or a 1-based pid includes one extra lane | zero-based: `offs < N` |
| `P = N // B` | tail elements never launch | `P = cdiv(N, B)` |
| grid too small | unwritten tail | check `P * B >= N` |
| grid too large, no mask | duplicate writes, last writer wins | mask every store |
| swapped pid axes | tile assignment transposed, wrong or out-of-bounds addresses | keep `(Pm, Pn)` order host and kernel |
| row mask only | last column tile crosses into the next row | mask rows **and** columns |
| 1-D vs 2-D flattening | `pm*BM + pn*BN + j` is not `r*N + c` | `rows[:, None]*N + cols[None, :]` |
| `pid` treated as a thread | grid sized by elements, huge launch | one program = one CTA |
| `BLOCK` treated as threads | wrong `num_warps` reasoning | threads = `32*num_warps` |
| grid-stride step `B` | overlapping tiles, duplicate writes | step `num_programs(0)*B` |
| `num_programs` as work count | empty or extra iterations | compare with `cdiv(N, B)` |
| `p_m = t % G_n` | rows and columns swapped | `p_m = t // G_n`, `p_n = t % G_n` |

In [ ]:
# The pitfall arithmetic, checked on the chapter's smallest examples.
N, BLOCK = 10, 4
assert cdiv(N, BLOCK) == 3
assert (N // BLOCK) * BLOCK < N

offs = 2 * BLOCK + torch.arange(BLOCK)
assert (offs <= N).sum().item() == 3   # off-by-one includes N
assert (offs < N).sum().item() == 2    # correct mask

M, N, BM, BN = 5, 7, 2, 4
Pm, Pn = cdiv(M, BM), cdiv(N, BN)
assert (Pm, Pn) == (3, 2)
assert (Pn, Pm) == (2, 3)              # swapping axes changes the grid

cols = 1 * BN + torch.arange(BN)
assert cols[cols < N].tolist() == [4, 5, 6]
assert (2 * N + 3) == 17
assert (2 * BN + 3) != 17              # flatten with N, not BN
assert 32 * 4 == 128
assert 2 * BLOCK == 8
assert (3 & (3 - 1)) != 0              # 3 is not a power of two

## Self-check

1. For `$N=10$`, `$B=4$`, write the three rows of the pid/offs/mask table.
2. For `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=3$`, how many tiles are there, and how many elements does tile `$(2,2)$` own?
3. Why is `P = N // B` wrong, and what is the correct expression?
4. In a 2-D grid, which axis is the row tile? What happens if you swap them?
5. Convert `$t=7$` to `$(p_m,p_n)$` for a `$3\times3$` tile grid.
6. How many CUDA threads execute a program with `BLOCK=256` and `num_warps=4`?
7. Give an example where two programs may read the same tensor safely, and one where they may not write it.
8. What is the grid-stride step, and why is it `$PB$` rather than `$B$`?

## Summary

- `tl.program_id(axis)` is a scalar grid coordinate; `tl.program_id(0)` is not a thread id.
- A 1-D grid has `$P=\lceil N/B\rceil$` programs; 2-D and 3-D grids multiply their per-axis counts.
- A tile is owned by exactly one program; the mask is what makes a partial tile safe.
- `tl.num_programs(axis)` is the launch extent; `num_programs(0)*B` is the grid-stride step.
- A Triton program is a CTA: `num_warps` sets its threads, `BLOCK` sets its logical lanes.
- Every mapping is a choice; the two invariants are coverage and unique ownership.

**Where this leads.** [Chapter 3 — Triton Vector Construction](../ch03_vector_construction/03_complete.ipynb) turns these coordinates into vectors with `tl.arange`, `tl.full`, `tl.zeros`, and `tl.cast`.

## Exercises

- **Connection.** These exercises reuse the grids, masks, and mappings from Part I.

1. **Table drill.** Write the pid/offs/mask table for `$N=13$`, `BLOCK=5`. How many lanes are dead?
2. **2-D grid.** For `$M=9$`, `$N=11$`, `$B_M=4$`, `$B_N=3$`, give the grid and the row/column ranges of tile `$(2,1)$`.
3. **3-D grid.** For `$B=4$`, `$M=8$`, `$N=8$`, `$B_M=4$`, `$B_N=4$`, give the grid, the number of programs, and the base offset of program `$(b,p_m,p_n)=(2,1,1)$`.
4. **Inverse map.** For a `$4\times5$` tile grid (`$G_m=4$`, `$G_n=5$`), convert `$t=13$` to `$(p_m,p_n)$` and back.
5. **Grid too small.** Change `vec_add` to launch `N // BLOCK` programs. Which elements are stale, and why does the test catch it?
6. **Swapped axes.** Swap `program_id(0)` and `program_id(1)` in the transpose kernel without changing the launch. Describe the resulting address pattern and the first out-of-bounds access.
7. **Mask experiment.** Remove the column mask from the transpose kernel for `$M=5$`, `$N=7$`, `$B_M=2$`, `$B_N=4$`. Which tile writes past the row, and where does it land?
8. **Grid-stride limits.** Run the grid-stride kernel with `programs=1` and `programs=3` for `$N=10$`, `BLOCK=4`. What labels does each program write?
9. **Threads.** A kernel has `BLOCK=128`, `num_warps=8`. How many threads per CTA, and what is the average number of elements per thread? What changes if `num_warps=2`?
10. **Row-per-program.** Write a kernel with one program per row for an `$M\times N$` matrix and `BLOCK_N >= N`. Which mask is still required, and why is the grid exactly `$M$`?

Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_naive_grid()
print("CPU tests: OK")

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (test_vec_add_kernel, test_transpose_kernel, test_grid_stride_kernel):
    run_if_gpu(fn)